In [1]:
import numpy as np
import pandas as pd
import json

In [2]:
# this notebook pulls in information about each clinical trial from clinicaltrials.gov


In [3]:
import json

In [4]:
file_name = "./all_ctgov_cancer_trials_ever_9-3-25.json"
with open(file_name, encoding='utf-8') as json_file:
    trials_dict = json.load(json_file)

In [9]:
# Extract the relevant data
titles = []
brief_summaries = []
eligibility_criteria = []
nct_ids = []
statuses = []

for trial in trials_dict:
    try:
        titles.append(trial['protocolSection']['identificationModule']['officialTitle'])
    except:
        titles.append("")
    
    try:
        brief_summaries.append(trial['protocolSection']['descriptionModule']['briefSummary'])
    except:
        brief_summaries.append("")

    try:
        nct_ids.append(trial['protocolSection']['identificationModule']['nctId'])
    except:
        nct_ids.append("")

    try:
        eligibility_criteria.append(trial['protocolSection']['eligibilityModule']['eligibilityCriteria'])
    except:
        eligibility_criteria.append("")
        
    try:
        statuses.append(trial['protocolSection']['statusModule']['overallStatus'])
    except:
        statuses.append('')

        

In [11]:
                                    
trial_frame = pd.DataFrame({'nct_id': nct_ids, 
                            'title': titles,
                            'brief_summary': brief_summaries,
                            'eligibility_criteria': eligibility_criteria,
                            'status': statuses})

trial_frame['trial_text'] = trial_frame['title'] + "\n" + trial_frame['brief_summary'] + "\n" + trial_frame['eligibility_criteria']

In [12]:
trial_frame.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74932 entries, 0 to 74931
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   nct_id                74932 non-null  object
 1   title                 74932 non-null  object
 2   brief_summary         74932 non-null  object
 3   eligibility_criteria  74932 non-null  object
 4   status                74932 non-null  object
 5   trial_text            74932 non-null  object
dtypes: object(6)
memory usage: 3.4+ MB


In [13]:
(trial_frame.nct_id == '').value_counts()

nct_id
False    74932
Name: count, dtype: int64

In [14]:
(trial_frame.title == '').value_counts()

title
False    74434
True       498
Name: count, dtype: int64

In [15]:
(trial_frame.brief_summary == '').value_counts()

brief_summary
False    74932
Name: count, dtype: int64

In [16]:
(trial_frame.eligibility_criteria == '').value_counts()

eligibility_criteria
False    74925
True         7
Name: count, dtype: int64

In [17]:
(trial_frame.status == '').value_counts()

status
False    74932
Name: count, dtype: int64

In [18]:
trial_frame.status.value_counts()

status
COMPLETED                39728
RECRUITING               14883
TERMINATED                9528
ACTIVE_NOT_RECRUITING     6457
NOT_YET_RECRUITING        4336
Name: count, dtype: int64

In [19]:
trial_frame = trial_frame[~(trial_frame.title == '')]

In [20]:
trial_frame.to_csv('all_ctgov_cancer_trials_ever_9-3-25.csv')

In [21]:
recruiting_trials = trial_frame[trial_frame.status == 'RECRUITING']

In [ ]:
import numpy as np
import pandas as pd
import json
from transformers import AutoTokenizer
import torch
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '6,7'
from vllm import LLM, SamplingParams



enrollments = pd.read_csv('all_ctgov_cancer_trials_ever_9-3-25.csv')


llm = LLM(model='openai/gpt-oss-120b', tensor_parallel_size = 2, download_dir = '../../meta_ai', gpu_memory_utilization=0.93, max_model_len=15000)


def summarize_trials_multi_cohort(eligibility_texts, llm):

    tokenizer = llm.get_tokenizer()
    prompts = []
    for trial in eligibility_texts:
        messages = [
            {'role':'system', 'content': """
        Reasoning: high.
        """},      
              
            {'role':'user', 'content': """
            You are an expert clinical oncologist with an encyclopedic knowledge of cancer and its treatments.
        Your job is to review a clinical trial document and extract a list of structured clinical spaces that are eligible for that trial.
        A clinical space is defined as a unique combination of cancer primary site, histology, which treatments a patient must have received, which treatments a patient must not have received, cancer burden (eg presence of metastatic disease), and tumor biomarkers (such as germline or somatic gene mutations or alterations, or protein expression on tumor) that a patient must have or must not have; that renders a patient eligible for the trial.
        Trials often specify that a particular treatment is excluded only if it was given within a short period of time, for example 14 days, one month, etc , prior to trial start. Do not include this type of time-specific treatment eligibility criteria in your output at all.
        Some trials have only one space, while others have several. Do not output a space that contains multiple cancer types and/or histologies. Instead, generate separate spaces for each cancer type/histology combination.
        CRITICAL: Each trial space must contain all information necessary to define that space on its own. It may not refer to other previously defined spaces for the same trial, since for later use, the spaces will be extracted and separated from each other. YOU MAY NOT include text describing a given space that refers to a previous space space; eg, "Same as above"-style output is not allowed!
        For biomarkers, if the trial specifies whether the biomarker will be assessed during screening, note that.
        Spell out cancer types; do not abbreviate them. For example, write "non-small cell lung cancer" rather than "NSCLC".
        Structure your output like this, as a list of spaces, with spaces separated by newlines, as below:
        1. Cancer type allowed: <cancer_type_allowed>. Histology allowed: <histology_allowed>. Cancer burden allowed: <cancer_burden_allowed>. Prior treatment required: <prior_treatments_requred>. Prior treatment excluded: <prior_treatments_excluded>. Biomarkers required: <biomarkers_required>. Biomarkers excluded: <biomarkers_excluded>.
        2. Cancer type allowed: <cancer_type_allowed>, etc.
        If a concept is not relevant, such as if there are no prior treatents required, simply output NA for that concept.
        After you output the trial spaces, output a newline, then the text "Boilerplate exclusions:", then another newline.
        Then, list exclusion criteria described in the trial text that are unrelated to the trial space definitions. Such exclusions tend to be common to clinical trials in general.
        Common boilerplate exclusion criteria include a history of pneumonitis, heart failure, renal dysfunction, liver dysfunction, uncontrolled brain metastases, HIV or hepatitis, and poor performance status. """ +  "Here is a clinical trial document: \n" + trial + "\n" + """Now, generate your list of the trial space(s), followed by any boilerplate exclusions, formatted as above.
            Do not provide any introductory, explanatory, concluding, or disclaimer text.
            Reminder: Treatment history is an important component of trial space definitions, but treatment history requirements that are described as applying only in a given period of time prior to trial treatment MUST BE IGNORED.
            CRITICAL: A given trial space MUST NEVER refer to another previously defined space. You must NEVER output text like "same as #1" or "same criteria as above." Instead, you MUST REPEAT all relevant critiera for new each space SO THAT IT STANDS ON ITS OWN. A user who later looks at the text for one space will not have  access to text for other spaces, and so output like "Same criteria as #1..." renders a space useless."""
            }
        ]
    
        prompts.append(tokenizer.apply_chat_template(conversation=messages, add_generation_prompt=True, tokenize=False))
    

    
    responses = llm.generate(
        prompts,   
        SamplingParams(
        temperature=0.0,
            top_k=1,
        max_tokens=10000,
        repetition_penalty=1.3
        #stop_token_ids=[tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|eot_id|>")],  # KEYPOINT HERE
    ))

    response_texts = [x.outputs[0].text for x in responses]

    reasoning_marker = "assistantfinal" # Adjust if your model uses different reasoning markers
    boilerplate_marker = "Boilerplate exclusions:"

    llm_output = []
    
    for response_text in response_texts:
        if reasoning_marker in response_text:
            llm_output.append(response_text.split(reasoning_marker, 1)[-1])
        else:
            llm_output.append(response_text)

    boilerplate_output = []
    space_output = []
    for text in llm_output:
        if boilerplate_marker in text:
            boilerplate_output.append(text.split(boilerplate_marker, 1)[-1])
            space_output.append(text.split(boilerplate_marker, 1)[0])
        else:
            boilerplate_output.append(text)
            space_output.append(text)

    return response_texts, llm_output, space_output, boilerplate_output

trials = enrollments.groupby('protocol_number').first().reset_index()
trials.info()



trial_cohorts = summarize_trials_multi_cohort(trials.trial_text.tolist(), llm)

trials['space_reasoning_and_output'] = trial_cohorts[0]
trials['space_output_no_reasoning'] = trial_cohorts[1]
trials['space_text'] = trial_cohorts[2]
trials['trial_boilerplate_text'] = trial_cohorts[3]

trials.to_csv('all_ctgov_cancer_trial_spaces_ever_10-4-25.csv')

import pandas as pd
import numpy as np
trials = pd.read_csv('all_ctgov_cancer_trial_spaces_ever_10-4-25.csv')


trials.info()